Objectif : générer un prompt de génération de vidéo à partir des transcriptions des audios

In [2]:
import os
import time
import fal_client
import requests
from openai import OpenAI 
from dotenv import load_dotenv
from moviepy import VideoFileClip, concatenate_videoclips

In [ ]:
load_dotenv()
api_key = os.environ.get("OPENAI_KEY")
client = OpenAI(api_key=api_key)

def generate_visual_bible(full_story_text):
    """Génère une description fixe des personnages et du style pour TOUTE l'histoire. Permet de garder un contexte globale unique d'une partie à l'autre.
    Ce context seras mis en dans le prompt de la vidéo avant la description de la scène correspond à la partie en question.
    """
    print("🎨 Création de la Bible Visuelle pour cette histoire...")
    system_instruction = (
        "You are a Concept Artist. Read this story and define a consistent visual style "
        "and character descriptions. Describe the main characters' physical traits, "
        "clothing, and the overall animation style (e.g., 2D flat, 3D Pixar-like, etc.). "
        "Keep it to 3-4 sentences maximum in English. No fluff."
    )
    
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": f"Story summary: {full_story_text[:4000]}"} # On limite pour ne pas exploser les tokens
            ]
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Erreur lors de la création de la bible : {e}")
        return "High-quality 2D digital animation, vibrant colors, detailed characters."

def create_video_prompt(transcription_text, visual_bible, model="gpt-4o-mini"):
    """Génère le prompt d'un segment en utilisant la bible visuelle imposée."""
    system_prompt = (
        f"You are a Video Prompt Engineer. \n"
        f"VISUAL BIBLE TO FOLLOW: {visual_bible}\n\n"
        "Your task: Transform the user text into a SINGLE FLUID PARAGRAPH in English. "
        "You MUST keep the characters and style identical to the VISUAL BIBLE provided above. "
        "Focus on the specific actions, emotions, and camera movements for this scene. "
        "Include technical terms: '720p', 'high resolution', 'masterpiece', 'smooth motion'. "
        "No headers, no bullet points."
    )

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": transcription_text}
            ],
            max_completion_tokens=1000,
            temperature=0.7
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Erreur API OpenAI : {e}")
        return None

if __name__ == "__main__":
    input_base_folder = "../Transcriptions"
    output_base_folder = "../Prompts_generation_video"

    # On parcourt chaque dossier (une histoire = un dossier)
    for root, dirs, files in os.walk(input_base_folder):
        txt_files = [f for f in files if f.endswith(".txt")]
        if not txt_files:
            continue

        print(f"\n--- 📁 Traitement du dossier : {os.path.basename(root)} ---")
        
        # 1. Lire tout le texte du dossier pour créer la Bible Visuelle
        full_text = ""
        for filename in sorted(txt_files):
            with open(os.path.join(root, filename), "r", encoding="utf-8") as f:
                full_text += f.read() + " "
        
        visual_bible = generate_visual_bible(full_text)
        print(f"📖 Bible générée : {visual_bible}")

        # 2. Créer les prompts pour chaque fichier avec cette bible
        for filename in sorted(txt_files):
            input_path = os.path.join(root, filename)
            
            relative_path = os.path.relpath(root, input_base_folder)
            target_output_dir = os.path.join(output_base_folder, relative_path)
            os.makedirs(target_output_dir, exist_ok=True)

            with open(input_path, "r", encoding="utf-8") as f:
                text = f.read().strip()

            if not text: continue

            print(f"⏳ Prompt segment : {filename}...")
            prompt_eng = create_video_prompt(text, visual_bible)

            if prompt_eng:
                out_filename = f"{os.path.splitext(filename)[0]}_prompt.txt"
                out_path = os.path.join(target_output_dir, out_filename)
                with open(out_path, "w", encoding="utf-8") as fw:
                    fw.write(prompt_eng)
            
            time.sleep(0.5)

    print("\n✨ Terminé ! Tes prompts sont maintenant cohérents par dossier.")


--- 📁 Traitement du dossier : A one minute story|Short Stories|A one minute story in English#Shortstoriesenglish #oneminutestories ---
🎨 Création de la Bible Visuelle pour cette histoire...
📖 Bible générée : Visual Style: The animation style will be vibrant and colorful 2D flat with bold outlines, reminiscent of classic children's storybooks. 

Character Descriptions: The rabbit is a plump, fluffy character with oversized ears, wide eyes, and a small twitching nose, wearing a simple green vest. The butterfly, with delicate, multicolored wings and a cheerful expression, contrasts the rabbit's panic, showcasing whimsy in her design. The other animals will feature exaggerated features and soft textures, emphasizing a playful and lighthearted theme throughout the story.
⏳ Prompt segment : A one minute story|Short Stories|A one minute story in English#Shortstoriesenglish #oneminutestories_part01.txt...
⏳ Prompt segment : A one minute story|Short Stories|A one minute story in English#Shorts

# Génération de vidéo à partir de modèle Hugging face

La génération de vidéo se déroule en deux étapes :

        1. La fusion de tous les parties de prompt en un seul prompt

        2. L'utilisation de fal-ai/hunyuan-video comme model de génération text to vidéo (le mieux noté sur Hugging Face) 

            -> On utilise le modèle via l'API de la plateforme fal.ai spécifique de la génération de contenue, plus facile à mettre en place que sur Hugging Face.

In [5]:
load_dotenv()

# --- CONFIGURATION DE COHÉRENCE ---
# On utilise la même graine pour tous les clips d'une même histoire
STORY_SEED = 42 

def download_video(url, save_path):
    print(f"📥 Téléchargement vers : {save_path}")
    response = requests.get(url)
    with open(save_path, "wb") as f:
        f.write(response.content)

def generate_segment(prompt_text, part_index, output_dir):
    """Génère un clip de 129 frames avec une graine fixe pour la cohérence."""
    
    video_filename = os.path.join(output_dir, f"part_{part_index:02d}.mp4")
    
    # SÉCURITÉ : Si le fichier existe déjà, on ne le régénère pas (économie de crédits)
    if os.path.exists(video_filename):
        print(f"⏭️ Partie {part_index} déjà présente, on passe à la suite.")
        return video_filename

    print(f"🎬 Envoi Fal.ai - Partie {part_index}...")
    
    handler = fal_client.submit(
        "fal-ai/hunyuan-video",
        arguments={
            "prompt": prompt_text,
            "video_size": "720p_portrait",
            "num_frames": 129, # Limite stricte autorisée
            "fps": 24,
            "seed": STORY_SEED, # CLÉ DE LA COHÉRENCE VISUELLE
            "guidance_scale": 7.0, # On force le respect du prompt de la Bible Visuelle
            "negative_prompt": "blurry, distorted, low quality, 3D render, realistic, text, watermark"
        }
    )
    
    result = fal_client.result("fal-ai/hunyuan-video", handler.request_id)
    video_url = result['video']['url']
    
    download_video(video_url, video_filename)
    return video_filename

def assemble_video(video_files, final_name):
    print(f"🏗️ Montage final en cours ({len(video_files)} segments)...")
    # On charge les clips en s'assurant qu'ils sont valides
    clips = [VideoFileClip(f) for f in video_files]
    
    # On assemble les clips
    final_clip = concatenate_videoclips(clips, method="compose")
    
    # Exportation (codec standard pour les réseaux sociaux)
    final_clip.write_videofile(final_name, codec="libx264", audio_codec="aac")
    print(f"✅ Vidéo finale sauvegardée : {final_name}")

if __name__ == "__main__":
    # 1. Sélection de l'histoire (doit correspondre au nom du dossier de prompts)
    story_name = "The Goose and Its Golden Egg | Moral Stories | Animated Stories" 
    
    prompt_folder = f"../Prompts_generation_video/{story_name}"
    output_folder = f"../Final_Videos/{story_name}"
    os.makedirs(output_folder, exist_ok=True)

    # 2. Récupération des prompts (triés par part01, part02...)
    prompt_files = sorted([f for f in os.listdir(prompt_folder) if f.endswith(".txt")])
    
    if not prompt_files:
        print(f"❌ Aucun prompt trouvé dans {prompt_folder}")
    else:
        generated_clips = []

        for i, filename in enumerate(prompt_files, start=1):
            filepath = os.path.join(prompt_folder, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                prompt_content = f.read().strip()
            
            # Génération du clip
            try:
                clip_path = generate_segment(prompt_content, i, output_folder)
                generated_clips.append(clip_path)
            except Exception as e:
                print(f"💥 Erreur sur la partie {i} : {e}")

        # 3. Fusionner tous les clips réussis
        if generated_clips:
            final_output = os.path.join(output_folder, "HISTOIRE_COMPLETE.mp4")
            assemble_video(generated_clips, final_output)

🎬 Envoi Fal.ai - Partie 1...
📥 Téléchargement vers : ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/part_01.mp4
🎬 Envoi Fal.ai - Partie 2...
📥 Téléchargement vers : ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/part_02.mp4
🎬 Envoi Fal.ai - Partie 3...
📥 Téléchargement vers : ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/part_03.mp4
🎬 Envoi Fal.ai - Partie 4...
📥 Téléchargement vers : ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/part_04.mp4
🏗️ Montage final en cours (4 segments)...
MoviePy - Building video ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/HISTOIRE_COMPLETE.mp4.
MoviePy - Writing video ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/HISTOIRE_COMPLETE.mp4



frame_index:  24%|██▍       | 126/516 [00:01<00:05, 67.33it/s, now=None]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/part_01.mp4, 2764800 bytes wanted but 0 bytes read at frame index 129 (out of a total 129 frames), at time 5.38/5.38 sec. Using the last valid frame instead.
  warnings.warn(


MoviePy - Done !
MoviePy - video ready ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/HISTOIRE_COMPLETE.mp4
✅ Vidéo finale sauvegardée : ../Final_Videos/The Goose and Its Golden Egg | Moral Stories | Animated Stories/HISTOIRE_COMPLETE.mp4
